In [95]:
import pandas as pd
import numpy as np

# pip install category-encoders
import category_encoders as ce

In [ ]:
# it seems the data doesn't follow the UTF-8 format
# let's force the encoding to follow Latin1 (or ISO-8859-1)
df = pd.read_csv("../OnlineRetail.csv", encoding="ISO-8859-1")

# let's add a small interaction feature => multiplication
df['Sales'] = df['Quantity'] * df['UnitPrice']
df


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Sales
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom,20.34
...,...,...,...,...,...,...,...,...,...
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,12/9/2011 12:50,0.85,12680.0,France,10.20
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,12/9/2011 12:50,2.10,12680.0,France,12.60
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,12/9/2011 12:50,4.15,12680.0,France,16.60
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,12/9/2011 12:50,4.15,12680.0,France,16.60


In [97]:
# right, we seem to have over 4000 unique options
df['Description'].nunique()

4223

**Example 1: Target encoding (replace each unique category with that category's average)**

In [98]:
# create a target encoder by using category encoders
encoder = ce.TargetEncoder(cols=["Description"])

# apply the encoder, Description is going to be encoded, Sales is our target variable
df_encoded_target = encoder.fit_transform(df['Description'], df['Sales'])

# insert the encoded description averages back to the original data
df['Description_encoded'] = df_encoded_target

# the usual process is this => you save and use the original product name
# => then you get the matching average from a separate file (which we created based on the data)
# => right before you use the ML model to predict the sales => drop the original product name
# df.drop("Description", axis=1)
df.head(5)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Sales,Description_encoded
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom,15.30,42.071959
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom,20.34,24.136433
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom,22.00,25.579590
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom,20.34,34.050825
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom,20.34,48.749510


In [99]:
# based on target encoding, average price for WHITE METAL LANTERN
df_test = df[df['Description'] == "WHITE METAL LANTERN"]
df_test['Sales'].mean().item()

# Sales could be our target variable
# for each category in description
# we replace the text category with its average of the sales value

24.13643292682927

**If data overfits easily with ML model and the target encoded data (average), you can use LeaveOneOutEncoder instead to reduce overfitting (adds a bit of variance)**

In [100]:
# create a target encoder by using category encoders
encoder = ce.LeaveOneOutEncoder(cols=["Description"])

# apply the encoder, Description is going to be encoded, Sales is our target variable
df_encoded_target = encoder.fit_transform(df['Description'], df['Sales'])

# insert the encoded description averages back to the original data
df['Description_LOO_encoded'] = df_encoded_target

# the usual process is this => you save and use the original product name
# => then you get the matching average from a separate file (which we created based on the data)
# => right before you use the ML model to predict the sales => drop the original product name
# df.drop("Description", axis=1)
df.head(5)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Sales,Description_encoded,Description_LOO_encoded
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom,15.30,42.071959,42.083264
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom,20.34,24.136433,24.148043
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom,22.00,25.579590,25.591849
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom,20.34,34.050825,34.079873
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom,20.34,48.749510,48.812924


In [101]:
# Leave One Out-encoder is almost the same as 
# Target Encoding, BUT for each row in the dataset
# and the average target calculation, THAT ONE PARTICULAR
# row is excluded from the calculation
category_row_sales = {
    2,
    5,
    #7,
    2,
    5,
    1
}

# quick average calculation
average = sum(category_row_sales) / len(category_row_sales)
average

2.6666666666666665

**Example 2 - Frequency (count) encoder => replace each unique category with the frequencty/count of that category (probably measures popularity of a category quite well, if that's important for your ML model)**

In [102]:
# quick check, top 5 value counts of in Description
df['Description'].value_counts().head(5)

Description
WHITE HANGING HEART T-LIGHT HOLDER    2369
REGENCY CAKESTAND 3 TIER              2200
JUMBO BAG RED RETROSPOT               2159
PARTY BUNTING                         1727
LUNCH BAG RED RETROSPOT               1638
Name: count, dtype: int64

In [103]:
# create a target encoder by using category encoders
encoder = ce.CountEncoder(cols=["Description"])

# apply the encoder, Description is going to be encoded, Sales is our target variable
df_encoded_target = encoder.fit_transform(df['Description'])

# insert the encoded description averages back to the original data
df['Description_frequency_encoded'] = df_encoded_target

# the usual process is this => you save and use the original product name
# => then you get the matching average from a separate file (which we created based on the data)
# => right before you use the ML model to predict the sales => drop the original product name
# df.drop("Description", axis=1)
df[['Description', 'Sales', 'Description_frequency_encoded']].head(5)

# REMEMBER: if you drop the original description and after that remove all duplicates
# you will most likely have less rows in the data because of that

# the frequency/count approach is probably most effective when
# the popularity of some certain item has a meaning in ML model
# for ales data, popular items probably have more optimistic sales estimations
# (etc. depends on context/situation however)

,Description,Sales,Description_frequency_encoded
0,WHITE HANGING HEART T-LIGHT HOLDER,15.30,2369
1,WHITE METAL LANTERN,20.34,328
2,CREAM CUPID HEARTS COAT HANGER,22.00,293
3,KNITTED UNION FLAG HOT WATER BOTTLE,20.34,473
4,RED WOOLLY HOTTIE WHITE HEART.,20.34,449


**Example 3 - Hash encoding / feature hashing (or hashing trick)**

In [ ]:
# let's save this for studying the hashing buckets after the operation
# HashingEncoder will remove the original category after being used
df['Description_original'] = df['Description']

In [ ]:
# create a target encoder by using category encoders
encoder = ce.HashingEncoder(cols=['Description'], n_components=6)

# apply the encoder, Description is going to be encoded, Sales is our target variable
df_encoded_hash = encoder.fit_transform(df)

# we have now simplified 4000+ categories into 6
# consult Google/your favorite generative AI for ideas on how 
# to evaluate the quality of the hashing and the amount of buckets
df_encoded_hash

,col_0,col_1,col_2,col_3,col_4,col_5,InvoiceNo,StockCode,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Sales,Description_encoded,Description_LOO_encoded,Description_frequency_encoded,Description_original
0,1,0,0,0,0,0,536365,85123A,6,12/1/2010 8:26,2.55,17850.0,United Kingdom,15.30,42.071959,42.083264,2369,WHITE HANGING HEART T-LIGHT HOLDER
1,0,0,1,0,0,0,536365,71053,6,12/1/2010 8:26,3.39,17850.0,United Kingdom,20.34,24.136433,24.148043,328,WHITE METAL LANTERN
2,0,0,0,1,0,0,536365,84406B,8,12/1/2010 8:26,2.75,17850.0,United Kingdom,22.00,25.579590,25.591849,293,CREAM CUPID HEARTS COAT HANGER
3,0,0,0,1,0,0,536365,84029G,6,12/1/2010 8:26,3.39,17850.0,United Kingdom,20.34,34.050825,34.079873,473,KNITTED UNION FLAG HOT WATER BOTTLE
4,0,1,0,0,0,0,536365,84029E,6,12/1/2010 8:26,3.39,17850.0,United Kingdom,20.34,48.749510,48.812924,449,RED WOOLLY HOTTIE WHITE HEART.
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
541904,0,1,0,0,0,0,581587,22613,12,12/9/2011 12:50,0.85,12680.0,France,10.20,8.203473,8.189864,148,PACK OF 20 SPACEBOY NAPKINS
541905,0,0,0,0,1,0,581587,22899,6,12/9/2011 12:50,2.10,12680.0,France,12.60,13.493844,13.496646,320,CHILDREN'S APRON DOLLY GIRL
541906,0,0,1,0,0,0,581587,23254,4,12/9/2011 12:50,4.15,12680.0,France,16.60,21.823322,21.840392,307,CHILDRENS CUTLERY DOLLY GIRL
541907,1,0,0,0,0,0,581587,23255,4,12/9/2011 12:50,4.15,12680.0,France,16.60,25.086538,25.139255,162,CHILDRENS CUTLERY CIRCUS PARADE


In [106]:
df_encoded_hash.columns


Index(['col_0', 'col_1', 'col_2', 'col_3', 'col_4', 'col_5', 'InvoiceNo',
       'StockCode', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID',
       'Country', 'Sales', 'Description_encoded', 'Description_LOO_encoded',
       'Description_frequency_encoded', 'Description_original'],
      dtype='object')

In [ ]:
# let's quickly see what kind of products were placed in the first "cluster" / bucket
df_encoded_hash.query('col_0 == 1')['Description_original']

0         WHITE HANGING HEART T-LIGHT HOLDER
8                  HAND WARMER RED POLKA DOT
21                  JAM MAKING SET WITH JARS
22               RED COAT RACK PARIS FASHION
44                   MINI PAINT SET VINTAGE 
                         ...                
541870         ASSORTED BOTTLE TOP  MAGNETS 
541886            MAGNETS PACK OF 4 SWALLOWS
541889        FAIRY TALE COTTAGE NIGHT LIGHT
541907       CHILDRENS CUTLERY CIRCUS PARADE
541908         BAKING SET 9 PIECE RETROSPOT 
Name: Description_original, Length: 82826, dtype: object

In [ ]:
# there's still plenty of different encoders in the category encoders
# -module, feel free to experiment!